In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-groq
!pip install langchain-google-genai
!pip install langchain-openai
!pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

## Structured Output

Models can be requested to provide their reponse in a structured way or in a format matching a given schema. This is useful for ensuring the output can be easily parsed and further processed. Langchain provides multiple schemas and methods to force structured outputs

## Pydantic

Pydantic models provide the richest feature set with field validation, nested structure, descriptions.

## Field Validation - Pydantic

Basically Field is a thing which representing a variable and provides us different parameters we can set for field validation

In [ ]:
from pydantic import BaseModel, Field # This Field is something we are going to use for our field validation and BaseModel is the base class for creating pydantic model

class Movie(BaseModel):
  title:str = Field(description="This is the title of the movie") # Here this title varaible must contain a str value and is of a field type
  year:int = Field(description="This year the movie was released")
  director:str = Field(description="Name of the director of the movie")
  rating:float = Field(description="Rating of the movie out of 10")

This description is very useful for the LLM because it makes LLM know like in which field it needs to place the output coming

Now to Attach or bind this structure with our model we use with_structured_output() and pass our class as a parameter

In [ ]:
structure_model = model.with_structured_output(Movie)
structure_model

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x79d3df34f5c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x79d3cc81cc50>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties'

Now Let's test it with our normal model and with structure_model to see how output differs

In [ ]:
model.invoke("Tell me details of the movie intersteller")

AIMessage(content='Interstellar is a 2014 science fiction film directed by Christopher Nolan, based on a screenplay written by Nolan and his brother Jonathan Nolan. The film\'s visual effects and production design were created by a team of over 200 people, and it was a major production that took several years to complete.\n\n**Plot**\n\nThe movie takes place in a dystopian future where Earth is facing a severe water crisis. The planet\'s crops are dying due to a phenomenon known as the "blight," caused by a massive reduction in oxygen levels in the atmosphere. As a result, humanity is on the brink of extinction.\n\nA team of scientists, led by Professor Brand (Michael Caine), proposes a plan to save humanity by traveling through a wormhole in search of a new habitable planet. The wormhole is believed to lead to three possible planets that can sustain human life.\n\nThe film follows the story of Cooper (Matthew McConaughey), a former NASA pilot who is recruited to lead the mission to th

In [ ]:
structure_model.invoke("Tell me details of the movie intersteller")

Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.2)

So Basically we can use this information easily for further processing but with normal model it is giving me all the information it learned through its training on the data

We can also display the raw message generated also we just have to pass a parameter "include_raw = True" while binding our model with our pydantic structure

In [ ]:
structure_model_with_raw = model.with_structured_output(Movie, include_raw=True)
structure_model_with_raw.invoke("Tell me details of the movie intersteller")

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '7kdpdt0qj', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.1,"title":"Interstellar","year":2014}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 285, 'total_tokens': 317, 'completion_time': 0.042010323, 'completion_tokens_details': None, 'prompt_time': 0.105942069, 'prompt_tokens_details': None, 'queue_time': 0.042078631, 'total_time': 0.147952392}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f2808-825c-77d3-bcbc-181f28605a92-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.1, 'title': 'Interstellar', 'year': 2014}, 'id': '7kdpdt0qj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 285, 'output_tokens': 

One more thing like we have defined the title should be a str and if by chance a int or other type of value comes it is going to give an error that type of Field validation is auto supported in pydantic

## Nested Structure - Pydantic

We can have different actors in a movie or we can a list of actors, so we can easily define them with the help of nested structure

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
  name:str
  role:str
  age:int
  budget:float | None # Here this is how we can type float or None anything from both because budget of some actors may not be revealed

class MovieDetails(BaseModel):
  title:str
  year:str
  rating:float
  actors:list[Actor] # This is nested structure
  genre:list[str] # This how we can define string

model_with_nested_structure = model.with_structured_output(MovieDetails)
response = model_with_nested_structure.invoke("Tell me details of the movie intersteller")
response

MovieDetails(title='Interstellar', year='2014', rating=8.1, actors=[Actor(name='Matthew McConaughey', role='Cooper', age=48, budget=165000000.0), Actor(name='Anne Hathaway', role='Brand', age=38, budget=165000000.0), Actor(name='Jessica Chastain', role='Murph', age=46, budget=165000000.0)], genre=['Science Fiction', 'Adventure', 'Drama'])

We can see actors field is a list of Actor class which contains all the fields with their values we defined in it, also genre is list of str values

In [ ]:
response.actors[1].name # This is how we can access a particular value

'Anne Hathaway'

## TypedDict

TypedDict provide a simple alternative using python built-in typing, ideal when you dont need runtime validation

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
  title: Annotated[str, ..., "This is the title of the movie"] # These "..." are also metadata
  year: Annotated[int, ..., "Year of release of the movie"]
  director: Annotated[str, ..., "Name of the director of movie"]
  rating: Annotated[float, ..., "Rating of the movie out of 10"]

model_with_Type_Dict = model.with_structured_output(MovieDict)
model_with_Type_Dict.invoke("Tell me details of the movie intersteller")

The above code cell might not run because Groq maybe not capable for strucctured output with TypedDict and it prefers Pydantic, This is just for learning and may work with OpenAI or Gemini model

These "..." are metadata here but are generally Ellipsis object in python many libraries treat it differently like in case of pydantic
:
name : str = ... #This means this is a required field
name : str | None #This means optional

So ... depends on library but many library utilise it as required

Also these metadata doesn't need to be string it can be any object, the library decides what to do with them..

Nested Structures using TypedDict

In [ ]:
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
  name:str
  role:str
  age:int
  budget:float | None # Here this is how we can type float or None anything from both because budget of some actors may not be revealed

class MovieDetails(TypedDict):
  title:str
  year:str
  rating:float
  actors:list[Actor] # This is nested structure
  genre:list[str] # This how we can define string

model_with_TypedDict_nested_structure = model.with_structured_output(MovieDetails)
response = model_with_TypedDict_nested_structure.invoke("Tell me details of the movie intersteller")
response

Same thing here also for Groq it might not work

## Let's see how to integrate structured output with our agents

In [ ]:
from pydantic import BaseModel, Field

from langchain.agents import create_agent

class ContactInfo(BaseModel):
  name:str = Field(description="Name of the person")
  email:str = Field(description="Email of the person")
  phone:str = Field(description="Phone number of the person")

agent = create_agent(
    model = model,
    response_format=ContactInfo   # We will pass the name of our structure in this variable and it auto selects the ProviderStrategy
)

result = agent.invoke({
    "messages" : [
        {
            "role" : "user",
            "content" : "Extract Contact info from the following information John Doe, johndoe@example.com, (555) 123-4567"
        }
    ]
})

result

{'messages': [HumanMessage(content='Extract Contact info from the following information John Doe, johndoe@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='b2151c99-1a8a-42ce-89d8-e568787888d1'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4havas52r', 'function': {'arguments': '{"email":"johndoe@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 277, 'total_tokens': 313, 'completion_time': 0.038125544, 'completion_tokens_details': None, 'prompt_time': 0.018914327, 'prompt_tokens_details': None, 'queue_time': 0.005908789, 'total_time': 0.057039871}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f282b-ace7-75a3-bb85-101293a6f076-0', tool_calls=[{'name': 'Contac

In [ ]:
result["structured_response"]

ContactInfo(name='John Doe', email='johndoe@example.com', phone='(555) 123-4567')

We can also do this with TypedDict, the difference is only Validations are not applied there

## DataClasses

A data class is a class typically containing mainly data. There are not any restrictions though, we can create it using @dataclass decorator

Let's see how we can integrate dataclass with agent for structured output

In [ ]:
from dataclasses import dataclass

from langchain.agents import create_agent

@dataclass
class ContactInfo():
  name:str
  email:str
  phone:str


dataclass_agent = create_agent(
    model = model,
    response_format=ContactInfo
)

dataclass_result = dataclass_agent.invoke({
    "messages":[
        {
            "role": "user",
            "content": "Extract Contact info from the following information John Doe, johndoe@example.com, (555) 123-4567"
        }
    ]
})


dataclass_result

{'messages': [HumanMessage(content='Extract Contact info from the following information John Doe, johndoe@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='1c233ac9-6a38-4899-8802-b327401383bd'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'r0vnrngh9', 'function': {'arguments': '{"email":"johndoe@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 283, 'total_tokens': 319, 'completion_time': 0.035270594, 'completion_tokens_details': None, 'prompt_time': 0.084361997, 'prompt_tokens_details': None, 'queue_time': 0.005964716, 'total_time': 0.119632591}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f282b-c140-7fd1-a791-26854dc96927-0', tool_calls=[{'name': 'Contac

In [ ]:
dataclass_result["structured_response"]

ContactInfo(name='John Doe', email='johndoe@example.com', phone='(555) 123-4567')

We can do the same this with DataClass Also and dataclass is there in python since its 3.7 version

In dataclass also we dont have validations like runtime validation and features like automatic type conversions, Field descriptions, Serialization, etc.

Serialization in pydantic is converting a model object into a standard format like dictionary or json so it can be saved in a file, stored in a database, sent over network, etc.

we use fucntions
- model.model_dump() - Converts to dictionary
- model.model_dump_json() - Converts to json

In [ ]:
result["structured_response"].model_dump()

{'name': 'John Doe', 'email': 'johndoe@example.com', 'phone': '(555) 123-4567'}

In [ ]:
result["structured_response"].model_dump_json()

'{"name":"John Doe","email":"johndoe@example.com","phone":"(555) 123-4567"}'

Note that this "result" variable is from pydantic structure we used with our agent in previous code cell they are not compatible with dataclass as we talked above so it wont work with "dataclass_result"

For dataclass we have to use asdict(function)

In [ ]:
from dataclasses import asdict

dict_result_dataclass = asdict(dataclass_result["structured_response"])
dict_result_dataclass

{'name': 'John Doe', 'email': 'johndoe@example.com', 'phone': '(555) 123-4567'}